# Kalshi Bot v2 — full pipeline

Thin notebook. All logic lives in `kalshi_v2/`. Cells here just
drive the lifecycle (build clients → start bot → inspect → stop)
and print results inline.

**Prereq.** `~/.kalshi/credentials.env` must contain:
```
KALSHI_PROD_KEY_ID=...
KALSHI_PROD_PRIVATE_KEY_PATH=~/.kalshi/prod_private_key.pem
```

**Pipeline.** Coinbase BTC spot → Kalshi WS orderbooks → empirical
bank fair value (drift-removed, vol+kurt matched) → HRDNN P-robust
filter (16 bootstrapped measures) → Lipschitz-clamped sizing →
risk preflight → place / record.

In [1]:
# Install / upgrade websockets in this kernel (v13+ uses
# additional_headers; <=10.x uses extra_headers — kalshi_v2/data.py
# auto-detects whichever is available, but v13+ is preferred).
%pip install -q 'websockets>=13' cryptography requests pandas numpy scipy python-dateutil
print('deps ok — RESTART KERNEL if websockets was upgraded, then re-run from cell 1')

Note: you may need to restart the kernel to use updated packages.
deps ok — RESTART KERNEL if websockets was upgraded, then re-run from cell 1


## 1. Setup

In [2]:
%load_ext autoreload
%autoreload 2

import time
from datetime import datetime, timezone

from kalshi_v2.config import CFG
from kalshi_v2.client import KalshiClient
from kalshi_v2 import data as v2data
from kalshi_v2.data import (fetch_historical_minutes, add_rv_features,
                              SPOT, BOOKS, TRACKED, WS_STATE, BOT_STATE)
from kalshi_v2.model import build_empirical_bank
from kalshi_v2.robust import build_ambiguity_set, SIZER
from kalshi_v2.strategy import scan_signals
from kalshi_v2.paper_db import open_trades, settled_trades
from kalshi_v2.risk import RISK_BLOCKS, get_live_balance
from kalshi_v2.portfolio import (paper_portfolio_metrics,
                                   live_portfolio_metrics, portfolio_metrics)
from kalshi_v2.main import (start_bot, stop_bot, status, kill_switch,
                              enable_live, disable_live, cancel_all_live_orders,
                              dashboard, tail_log)

print('imports OK')
print(f'mode:          {CFG["mode"]}')
print(f'live_enabled:  {CFG["live_enabled"]}')
print(f'robust_enabled: {CFG["robust_enabled"]}')
print(f'sfm_enabled:   {CFG["sfm_enabled"]}')

imports OK
mode:          paper
live_enabled:  False
robust_enabled: True
sfm_enabled:   False


## 2. Build clients

Two clients: `kalshi_md` for read-only market data (no auth required
for public endpoints) and `kalshi_live` for authed actions (orders,
balance, WS handshake).

In [3]:
kalshi_md   = KalshiClient(env='prod')
kalshi_live = KalshiClient(env='prod')

print(f'md client:    base={kalshi_md.base_url}, signed={kalshi_md.private_key is not None}')
print(f'live client:  base={kalshi_live.base_url}, signed={kalshi_live.private_key is not None}')
print(f'live key_id:  {kalshi_live.key_id[:8] + "..." if kalshi_live.key_id else None}')

if kalshi_live.private_key is None:
    print('\n  ⚠ live client unsigned — bot will run paper-only, no WS auth')

md client:    base=https://api.elections.kalshi.com/trade-api/v2, signed=True
live client:  base=https://api.elections.kalshi.com/trade-api/v2, signed=True
live key_id:  628659b8...


In [5]:
# Quick sanity check: hit a public endpoint
try:
    ev = kalshi_md.get_events(series_ticker='KXBTC', status='open', limit=3)
    print(f'public REST OK — {len(ev.get("events", []))} BTC events open')
    for e in ev.get('events', [])[:3]:
        print(f'  {e.get("event_ticker")}')
except Exception as e:
    print(f'REST sanity failed: {e}')

if kalshi_live.private_key is not None:
    try:
        bal = kalshi_live.get_balance()
        print(f'\nlive balance: ${float(bal.get("balance", 0))/100:.2f}')
    except Exception as e:
        print(f'\nbalance fetch failed: {e}')

public REST OK — 3 BTC events open
  KXBTC-26MAY1517
  KXBTC-26MAY0917
  KXBTC-26MAY0817

live balance: $180.00


## 3. Show config

In [7]:
from kalshi_v2.config import CFG
CFG['robust_min_pass_rate'] = 0.85

In [251]:
from kalshi_v2.config import RISK_PCT
RISK_PCT['max_per_trade'] = 0.20    # 20% of live balance per trade

In [252]:
for k in sorted(CFG.keys()):
    v = CFG[k]
    print(f'  {k:30s} {v}')

  arb_max_dollars_per_trade      1000
  bankroll                       100000.0
  db_path                        /Users/rithvikijju/.btc_kalshi_bot/v2.db
  decision_interval_sec          5.0
  event_series                   ('KXBTC', 'KXBTCD')
  i_acknowledge_real_money_risk  False
  kalshi_fee_cap                 0.07
  lipschitz_position_L           200
  live_enabled                   True
  max_concurrent_signals         1
  max_entry_price                0.8
  max_per_market                 0.02
  max_position_age_min           90
  max_spread_cents               3
  min_edge_cents                 2.5
  min_entry_price                0.2
  min_liquidity                  0
  min_model_confidence           0.0
  mode                           live
  order_buffer_cents             2
  order_expiration_sec           30
  rest_book_interval_sec         10
  robust_enabled                 True
  robust_min_mean_edge_c         1.0
  robust_min_pass_rate           0.85
  robust_n_bootstra

In [ ]:
CFG[]

## 4. Fetch BTC history + features

90 days of BTC 1-min bars from Coinbase. This populates the input to
`build_empirical_bank` and `build_ambiguity_set`. Takes 30–60 s.

In [11]:
btc_1m = add_rv_features(fetch_historical_minutes(days_back=90))
print(f'btc_1m: {len(btc_1m):,} bars')
print(f'  range:   {btc_1m["time"].min()}  →  {btc_1m["time"].max()}')
print(f'  spot:    ${btc_1m["close"].iloc[-1]:,.0f}')
print(f'  rv_60m last:  {btc_1m["rv_60m"].iloc[-1]:.4f} (annualized)')
btc_1m.tail(3)

btc_1m: 129,161 bars
  range:   2026-02-07 20:38:00+00:00  →  2026-05-08 20:37:00+00:00
  spot:    $80,223
  rv_60m last:  0.2405 (annualized)


,time,low,high,open,close,volume,log_ret,rv_5m,rv_15m,rv_60m,rv_240m,rv_1440m
129158,2026-05-08 20:35:00+00:00,80236.13,80257.09,80257.08,80237.09,0.702487,-0.000249,0.431169,0.300933,0.239857,0.301658,0.341358
129159,2026-05-08 20:36:00+00:00,80212.00,80237.10,80237.09,80227.87,1.190451,-0.000115,0.290342,0.269862,0.240260,0.301618,0.341152
129160,2026-05-08 20:37:00+00:00,80217.38,80249.29,80227.87,80222.65,6.617680,-0.000065,0.173140,0.269555,0.240466,0.301337,0.341149


## 5. Build empirical bank — preview before bot starts

Diagnostic: same call `start_bot` will make. Confirms drift removal
and conditioning features are sane.

In [12]:
bank = build_empirical_bank(btc_1m, horizon_min=60, n_samples=5000, demean=True)
R = bank['log_returns']
import numpy as np
print(f'  n samples:     {bank["n"]:,}')
print(f'  R mean (post-demean): {R.mean():+.6f}')
print(f'  R std:         {R.std():.6f}')
print(f'  R quantiles:   1%={np.quantile(R, 0.01):+.4f}, 50%={np.quantile(R, 0.50):+.4f}, 99%={np.quantile(R, 0.99):+.4f}')
print(f'  v_mean:        {bank["v_mean"]:.4f}')
print(f'  k_mean:        {bank["k_mean"]:+.3f}')

  n samples:     5,000
  R mean (post-demean): -0.000000
  R std:         0.004974
  R quantiles:   1%=-0.0138, 50%=-0.0001, 99%=+0.0140
  v_mean:        0.4234
  k_mean:        +2.105


## 6. Build ambiguity set — preview

16 bootstrapped empirical banks. The robust filter rejects a signal
unless **every** measure agrees the trade has positive post-fee edge.
Spread of v_mean across measures shows the filter has real ambiguity
to test against (vs a near-zero spread, which would mean it's a noop).

In [13]:
amb = build_ambiguity_set(btc_1m, horizon_min=60,
                            n_bootstrap=CFG['robust_n_bootstrap'])
v_means = [m['v_mean'] for m in amb]
k_means = [m['k_mean'] for m in amb]
print(f'  measures:        {len(amb)}')
print(f'  v_mean range:   [{min(v_means):.4f}, {max(v_means):.4f}]  (spread {max(v_means)-min(v_means):.4f})')
print(f'  k_mean range:   [{min(k_means):+.3f}, {max(k_means):+.3f}]  (spread {max(k_means)-min(k_means):.3f})')
if max(v_means) - min(v_means) < 0.01:
    print('  ⚠ low spread — bootstrap may not be giving ambiguity (degenerate ambiguity set)')
else:
    print('  ✓ measures vary — filter has something to test against')

  measures:        16
  v_mean range:   [0.4168, 0.4399]  (spread 0.0231)
  k_mean range:   [+1.999, +2.333]  (spread 0.335)
  ✓ measures vary — filter has something to test against


## 7. Start the bot

Spins up 4 daemon threads:
- `spot_poller` — Coinbase BTC spot every 2 s
- `ws_listener` — Kalshi WS orderbook stream (REST fallback on 401/403)
- `event_tracker` — finds nearest BTC event in TTL window every 60 s
- `decision`     — settle → manage → scan → execute every `decision_interval_sec`

Idempotent: re-running `start_bot` while running prints a warning and
no-ops. Pass `btc_1m=btc_1m, refresh_btc_1m=False` to skip the 90-day
refetch (we already have it).

In [334]:
start_bot(kalshi_md, kalshi_live, btc_1m=btc_1m, refresh_btc_1m=False)

  ✓ empirical bank: 5000 samples
  ✓ ambiguity set: 16 measures
  ✓ live balance: $100.29

✓ bot running (4 threads). mode=live


In [335]:
enable_live()

✓ LIVE TRADING ENABLED. balance=$100.29


## 8. Live status

Re-run this cell any time to see thread health, WS state, current
tracked event, recent log lines.

In [345]:
status(last_n_log_lines=20)

  V2 BOT STATUS @ 2026-05-11T14:36:58.407476+00:00
  mode:          live
  live_enabled:  True
  running:       True
  iter:          5
  trades:        0
  live balance:  $100.29

  Threads: 4
    v2_spot_poller             alive=True
    v2_ws_listener             alive=True
    v2_event_tracker           alive=True
    v2_decision                alive=True

  WebSocket:
    mode:          websocket
    connected:     True
    subscribed:    KXBTC-26MAY1111
    msgs received: 47247
    last msg:      2026-05-11 14:36:57.423117+00:00

  Tracked event:  KXBTC-26MAY1111
  Books in mem:   188

  Empirical bank: 5000
  Ambiguity set:  16 measures

  Log tail:
    [14:36:36] WS connected
    [14:36:36] tracking: KXBTC-26MAY1111 closes 2026-05-11 15:00:00+00:00
    [14:36:36] REST seed: 188 markets for KXBTC-26MAY1111
    [14:36:41] REST seed: 188 markets for KXBTC-26MAY1111
    [14:36:41] WS resubscribed: 188 tickers
    [14:36:41] WS sub confirmed sid=1
    [14:36:41] WS sub confirmed sid

## 8b. Full live dashboard

Re-runnable snapshot of the entire pipeline. Shows:
- threads + WS state
- spot, sigma, tracked event, sample books
- per-market edge breakdown for every market in scope, with the reason each one was rejected (no quote, spread too wide, edge too thin, entry out of band, robust filter rejection, etc.)
- last 10 robust filter decisions
- last 10 risk-preflight blocks
- open + settled trades + realized PnL
- recent log tail

## 8c. Stream the log

`tail_log(30)` prints the last 30 lines.
`tail_log(30, follow_secs=60)` blocks and streams new entries for 60s.

In [279]:
tail_log(30)
# Or stream live for a minute:
# tail_log(30, follow_secs=60)

[02:29:29] WS connected
[02:29:29] tracking: KXBTC-26MAY1023 closes 2026-05-11 03:00:00+00:00
[02:29:29] REST seed: 188 markets for KXBTC-26MAY1023
[02:29:34] REST seed: 188 markets for KXBTC-26MAY1023
[02:29:34] WS resubscribed: 188 tickers
[02:29:34] WS sub confirmed sid=1
[02:29:34] WS sub confirmed sid=2
[02:29:34] WS sub confirmed sid=3


## 9. Manual signal scan (diagnostic)

Calls the same `scan_signals` the decision worker calls, but inline
so you can see the edge distribution and which markets passed the
robust filter.

In [346]:
from kalshi_v2.main import _EMPIRICAL_BANK, _AMBIGUITY_SET

sigs = scan_signals(empirical_bank=_EMPIRICAL_BANK,
                      ambiguity_set=_AMBIGUITY_SET)
if len(sigs) == 0:
    print('no signals this scan')
    print(f'  spot:     {SPOT.get("price")}')
    print(f'  event:    {TRACKED.get("event")}')
    print(f'  books:    {len(BOOKS)}')
else:
    cols = ['ticker', 'side', 'entry_price', 'model_p_yes', 'edge_c',
            'robust_pass_rate', 'robust_mean_edge_c', 'ttl_min']
    cols = [c for c in cols if c in sigs.columns]
    print(f'{len(sigs)} signal(s) passed all filters:\n')
    print(sigs[cols].to_string(index=False))

no signals this scan
  spot:     80612.965
  event:    KXBTC-26MAY1111
  books:    188


## 10. Open positions

In [347]:
op = open_trades()
if len(op) == 0:
    print('no open positions')
else:
    cols = ['id', 'timestamp_utc', 'market_ticker', 'side', 'contracts',
            'entry_price', 'entry_edge_cents', 'model_p_yes', 'trade_type']
    cols = [c for c in cols if c in op.columns]
    print(f'{len(op)} open positions:\n')
    print(op[cols].to_string(index=False))

no open positions


## 11. Robust filter decisions log

Every signal that came through `scan_signals` (pass or fail) is
logged here. Useful for tuning `robust_min_pass_rate` and
`robust_min_mean_edge_c`.

In [348]:
from kalshi_v2.paper_db import _conn
import pandas as pd
conn = _conn()
rd = pd.read_sql_query(
    'SELECT ts, ticker, side, entry_price, n_measures, pass_rate, '
    'mean_edge_c, min_edge_c, max_edge_c, passed '
    'FROM robust_decisions ORDER BY ts DESC LIMIT 20', conn)
conn.close()
if len(rd) == 0:
    print('no robust decisions logged yet')
else:
    print(rd.to_string(index=False))

                              ts                     ticker side  entry_price  n_measures  pass_rate  mean_edge_c  min_edge_c  max_edge_c  passed
2026-05-11T02:42:57.544457+00:00     KXBTC-26MAY1023-B81150  yes         0.28          16     0.7500     1.844783   -3.478133    6.855200       0
2026-05-11T02:42:42.151118+00:00     KXBTC-26MAY1023-B81150  yes         0.26          16     0.8750     3.071133   -2.887200    8.612800       1
2026-05-11T02:42:36.973000+00:00     KXBTC-26MAY1023-B81150  yes         0.26          16     0.8750     2.998217   -3.053867    8.612800       1
2026-05-11T02:42:31.835829+00:00     KXBTC-26MAY1023-B81150  yes         0.26          16     0.6875     1.581550   -4.553867    8.446133       0
2026-05-11T02:42:26.637741+00:00     KXBTC-26MAY1023-B81150  yes         0.26          16     0.6250     1.498217   -4.720533    8.279467       0
2026-05-11T02:42:21.445620+00:00     KXBTC-26MAY1023-B81150  yes         0.26          16     0.9375     4.175300   -1.22053

## 12. Portfolio metrics

Paper view (CFG bankroll) and live view (Kalshi balance) are kept
separate. Open positions get marked-to-market via in-memory WS
books with REST fallback.

In [349]:
portfolio_metrics(kalshi_md=kalshi_md, kalshi_live=kalshi_live)

  PAPER PORTFOLIO
  bankroll: $    100,000.00  (CFG['bankroll'])
  time:     2026-05-11T14:37:10.187742+00:00

  Equity:           $    104,606.70  (+4.61%)
  Cash:             $    104,606.70
  Open cost:        $          0.00
  Realized:         $     +4,606.70
  Unrealized:       $         +0.00

  Settled: n=67  wins=38 (56.7%)

  LIVE / SHADOW PORTFOLIO
  bankroll: $        100.29  (live Kalshi balance)
  time:     2026-05-11T14:37:10.196670+00:00
  no trades.



## 13. Recent settled trades

In [320]:
st = settled_trades()
if len(st) == 0:
    print('no settled trades yet')
else:
    cols = ['id', 'market_ticker', 'side', 'contracts', 'entry_price',
            'settle_price', 'pnl_dollars', 'exit_reason', 'trade_type']
    cols = [c for c in cols if c in st.columns]
    print(f'{len(st)} settled trades, last 10:\n')
    print(st[cols].tail(10).to_string(index=False))
    print(f'\ntotal realized PnL: ${st["pnl_dollars"].sum():+.2f}')
    wins = (st['pnl_dollars'] > 0).sum()
    print(f'win rate: {wins}/{len(st)} = {wins/len(st)*100:.1f}%')

67 settled trades, last 10:

 id              market_ticker side  contracts  entry_price  settle_price  pnl_dollars           exit_reason trade_type
 58     KXBTC-26MAY1010-B80950   no       2924         0.64         0.500      -409.36     max_age (age=98m)         v2
 59 KXBTCD-26MAY1012-T80999.99  yes       3036         0.44         0.755       956.34  take_profit (+31.5c)         v2
 60 KXBTCD-26MAY1012-T81099.99  yes       3016         0.34         0.395       165.88   take_profit (+5.5c)         v2
 61 KXBTCD-26MAY1012-T80999.99   no       2994         0.21         0.295       254.49   take_profit (+8.5c)         v2
 62 KXBTCD-26MAY1012-T81099.99  yes       3008         0.30         0.970      2015.36  take_profit (+67.0c)         v2
 63 KXBTCD-26MAY1012-T81399.99   no       2928         0.66         0.855       570.96  take_profit (+19.5c)         v2
 64 KXBTCD-26MAY1012-T81299.99   no       2998         0.25         0.000      -749.50 resolution:result=yes         v2
 65 KXBTCD-

## 14. Risk-block log

Last 20 signals that were rejected by `risk_preflight`. Each entry
shows the reasons (ticker dedup, exposure cap, balance floor, etc.).

In [285]:
if not RISK_BLOCKS:
    print('no risk blocks recorded')
else:
    for rb in RISK_BLOCKS[-20:]:
        print(f'  {rb["ts"][:19]}  {rb["ticker"]:30s} {rb["side"]:>3s}  '
              f'{rb["strategy"]}  → {"; ".join(rb["reasons"])}')

  2026-05-10T05:03:58  KXBTCD-26MAY1002-T80799.99     yes  v2  → already open on KXBTCD-26MAY1002-T80799.99
  2026-05-10T05:04:03  KXBTCD-26MAY1002-T80799.99     yes  v2  → already open on KXBTCD-26MAY1002-T80799.99
  2026-05-10T06:33:58  KXBTCD-26MAY1003-T80599.99      no  v2  → already open on KXBTCD-26MAY1003-T80599.99
  2026-05-10T07:06:33  KXBTCD-26MAY1004-T80699.99      no  v2  → already open on KXBTCD-26MAY1004-T80699.99
  2026-05-10T07:15:55  KXBTCD-26MAY1004-T80699.99      no  v2  → already open on KXBTCD-26MAY1004-T80699.99
  2026-05-10T07:16:00  KXBTCD-26MAY1004-T80699.99      no  v2  → already open on KXBTCD-26MAY1004-T80699.99
  2026-05-10T07:16:14  KXBTCD-26MAY1004-T80699.99      no  v2  → already open on KXBTCD-26MAY1004-T80699.99
  2026-05-10T07:16:19  KXBTCD-26MAY1004-T80699.99      no  v2  → already open on KXBTCD-26MAY1004-T80699.99
  2026-05-10T07:22:59  KXBTCD-26MAY1004-T80799.99     yes  v2  → already open on KXBTCD-26MAY1004-T80799.99
  2026-05-10T08:30:01  KXBTC

## 15. Controls

Run any of these as needed.

In [292]:
# Stop the decision loop. Threads exit at next sleep wake (~250 ms).
# stop_bot()

# Hard kill: stop + force paper mode + refresh sessions.
# kill_switch()

# Flip mode to live. Refuses if balance unknown or $0.
enable_live()

# Flip back to paper.
# disable_live()

# Cancel every resting Kalshi order.
# cancel_all_live_orders()

✓ LIVE TRADING ENABLED. balance=$7.19


In [303]:
start_bot(kalshi_md, kalshi_live, btc_1m=btc_1m, refresh_btc_1m=False)

  ✓ empirical bank: 5000 samples
  ✓ ambiguity set: 16 measures
  ✓ live balance: $100.29

✓ bot running (4 threads). mode=paper


In [307]:
tail_log(30)

[02:55:02] WS connected
[02:55:03] REST seed: 188 markets for KXBTC-26MAY1023
[02:55:03] WS subscribed: 188 tickers × 3 channels
[02:55:03] WS sub confirmed sid=1
[02:55:03] WS sub confirmed sid=3
[02:55:03] WS sub confirmed sid=2
[02:55:03] tracker: 6 events found, 0 in TTL window (5-240 min)


In [352]:
enable_live()

✓ LIVE TRADING ENABLED. balance=$100.29


In [357]:
from kalshi_v2.robust import SIZER
SIZER.reset()   # clears all per-mode history; first new trade sizes from raw Kelly

In [369]:
from kalshi_v2.config import CFG
CFG['decision_interval_sec'] = 1.0   # was 5.0 — scan every second

In [380]:
dashboard()

  V2 DASHBOARD @ 2026-05-11 15:09:09

A. CONNECTIVITY
   threads alive:  4/4
     ✓ v2_spot_poller
     ✓ v2_ws_listener
     ✓ v2_event_tracker
     ✓ v2_decision
   ws connected:   True  (mode=websocket, msgs=56068)
   ws subscribed:  KXBTCD-26MAY1112

B. TRACKING
   spot:           $80,885.85
   causal sigma:   0.1253
   event:          KXBTCD-26MAY1112
   closes:         16:00 UTC  (ttl +50.8 min)
   books in mem:   202
   sample books:
     KXBTC-26MAY1112-B80050              yes_bid=0.0  yes_ask=0.04  floor=None
     KXBTC-26MAY1112-B80350              yes_bid=0.02  yes_ask=0.05  floor=None
     KXBTC-26MAY1112-B80450              yes_bid=0.03  yes_ask=0.06  floor=None

C. SCANNER (this snapshot, not the running worker)
   188 markets in event scope, 1 pass edge filters
     KXBTCD-26MAY1112-T81099.99        no entry=0.790 edge=+2.6c  · PASSES edge filters
     KXBTCD-26MAY1112-T81199.99        no entry=0.870 edge=+2.4c  · edge +2.4c < 2.5c
     KXBTCD-26MAY1112-T80699.99       y

In [378]:
portfolio_metrics()

  PAPER PORTFOLIO
  bankroll: $    100,000.00  (CFG['bankroll'])
  time:     2026-05-11T15:07:17.170235+00:00

  Equity:           $    104,606.70  (+4.61%)
  Cash:             $    104,606.70
  Open cost:        $          0.00
  Realized:         $     +4,606.70
  Unrealized:       $         +0.00

  Settled: n=67  wins=38 (56.7%)

  LIVE / SHADOW PORTFOLIO
  bankroll: $        100.29  (live Kalshi balance)
  time:     2026-05-11T15:07:17.175079+00:00
  no trades.

  v2 LIVE FAILED KXBTCD-26MAY1112-T81099.99: no live quotes on KXBTCD-26MAY1112-T81099.99
  v2 LIVE FAILED KXBTCD-26MAY1112-T81099.99: no live quotes on KXBTCD-26MAY1112-T81099.99
  v2 LIVE FAILED KXBTCD-26MAY1112-T81099.99: no live quotes on KXBTCD-26MAY1112-T81099.99
  v2 LIVE FAILED KXBTCD-26MAY1112-T81099.99: no live quotes on KXBTCD-26MAY1112-T81099.99
  v2 LIVE FAILED KXBTCD-26MAY1112-T81099.99: no live quotes on KXBTCD-26MAY1112-T81099.99
  v2 LIVE FAILED KXBTCD-26MAY1112-T81099.99: no live quotes on KXBTCD-26MAY111

In [311]:
disable_live()

LIVE DISABLED. mode=paper.


In [332]:
stop_bot()

stopping bot... (threads daemon-exit at next sleep wake)


In [315]:
status()

  V2 BOT STATUS @ 2026-05-11T02:56:30.946665+00:00
  mode:          live
  live_enabled:  False
  running:       True
  iter:          18
  trades:        0
  live balance:  $100.29

  Threads: 4
    v2_spot_poller             alive=True
    v2_ws_listener             alive=True
    v2_event_tracker           alive=True
    v2_decision                alive=True

  WebSocket:
    mode:          websocket
    connected:     True
    subscribed:    KXBTC-26MAY1023
    msgs received: 42506
    last msg:      2026-05-11 02:56:30.148830+00:00

  Tracked event:  None
  Books in mem:   188

  Empirical bank: 5000
  Ambiguity set:  16 measures

  Log tail:
    [02:55:02] WS connected
    [02:55:03] REST seed: 188 markets for KXBTC-26MAY1023
    [02:55:03] WS subscribed: 188 tickers × 3 channels
    [02:55:03] WS sub confirmed sid=1
    [02:55:03] WS sub confirmed sid=3
    [02:55:03] WS sub confirmed sid=2
    [02:55:03] tracker: 6 events found, 0 in TTL window (5-240 min)
    [02:56:03] tracke

In [268]:
stop_bot()

stopping bot... (threads daemon-exit at next sleep wake)


In [267]:
from kalshi_v2.state import BOOKS, SPOT, TRACKED
from kalshi_v2.main import _EMPIRICAL_BANK
from kalshi_v2.model import fair_value
from kalshi_v2.data import causal_sigma_from_spot
from datetime import datetime, timezone
import pandas as pd

spot = SPOT.get('price')
sigma = causal_sigma_from_spot()
event = TRACKED.get('event')
ct = TRACKED.get('close_time')
ttl_min = (ct - datetime.now(timezone.utc)).total_seconds() / 60 if ct else None

print(f'spot=${spot:,.2f}  sigma={sigma:.4f}  event={event}  ttl={ttl_min:.1f}min')
print(f'expected 1-sigma move over ttl: ${spot * sigma * (ttl_min/525960)**0.5:,.0f}')
print()

rows = []
for tk, b in BOOKS.items():
    if not tk.startswith(event or ''): continue
    yb, ya = b.get('yes_bid'), b.get('yes_ask')
    fl, cap = b.get('floor'), b.get('cap')
    if fl is None or yb is None or ya is None: continue
    try: fl_f = float(fl)
    except: continue
    p_yes = fair_value(tk, spot, fl_f, cap, ttl_min, sigma, kurt=0.0,
                         empirical_bank=_EMPIRICAL_BANK)
    if p_yes is None: continue
    mid = (yb + ya) / 2
    rows.append({
        'ticker': tk[-15:],
        'floor': fl_f,
        'dist_$': fl_f - spot,
        'yb': yb, 'ya': ya,
        'mkt_mid': round(mid, 3),
        'model_p': round(p_yes, 3),
        'diff_c': round((p_yes - mid) * 100, 1),
        'spread_c': round((ya - yb) * 100, 1),
    })

df = pd.DataFrame(rows)
df['abs_dist'] = df['floor'].apply(lambda x: abs(x - spot))
print('Markets sorted by distance from spot (closest = most likely to have action):')
print(df.sort_values('abs_dist').head(20).drop(columns=['abs_dist']).to_string(index=False))
print()
print('Top 5 abs(model − market) divergences (where any edge would be):')
df['abs_diff'] = df['diff_c'].abs()
print(df.sort_values('abs_diff', ascending=False).head(5).drop(columns=['abs_dist','abs_diff']).to_string(index=False))

TypeError: unsupported format string passed to NoneType.__format__